In [ ]:
import os

# Get credentials from environment variables (set outside the notebook)
KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME')
KAGGLE_KEY = os.environ.get('KAGGLE_KEY')

if KAGGLE_USERNAME is None or KAGGLE_KEY is None:
    raise ValueError("Please set KAGGLE_USERNAME and KAGGLE_KEY as environment variables.")


In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [03:55<00:00, 23.7MB/s]



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp skin-cancer-mnist-ham10000.zip /content/drive/MyDrive/

In [ ]:
import os

# 1. إعداد مفاتيح كاجل
os.environ['KAGGLE_USERNAME'] = "leenaldayoub" # رح يشتغل تلقائياً من المفاتيح اللي ادخلتيها
os.environ['KAGGLE_KEY'] = "KGAT_3bcccfe52be52eedad296f0b3782e694"

# 2. تنزيل وفك الضغط فوراً بثواني
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/ham10000


Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
skin-cancer-mnist-ham10000.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
ArithmeticError

In [4]:
# 1. تنزيل وفك ضغط الداتا مباشرة في خطوة واحدة
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -o -q skin-cancer-mnist-ham10000.zip -d /content/ham10000

print(" تم تنزيل وفك ضغط البيانات ")

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [04:15<00:00, 21.8MB/s]

 تم تنزيل وفك ضغط البيانات 


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

# ==========================================
# 1. إعداد المسارات والجهاز (GPU)
# ==========================================
BASE_DIR = '/content/ham10000'
CSV_PATH = os.path.join(BASE_DIR, 'HAM10000_metadata.csv')

IMG_DIR_1 = os.path.join(BASE_DIR, 'HAM10000_images_part_1')
IMG_DIR_2 = os.path.join(BASE_DIR, 'HAM10000_images_part_2')

# التأكد من استخدام كرت الشاشة CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using Device: {device}")


# ==========================================
# 2. بناء فئة الـ Dataset لتجهيز الصور
# ==========================================
class HAM10000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['image_id']
        img_name = f"{img_id}.jpg" if not str(img_id).endswith('.jpg') else img_id

        # البحث عن مكان الصورة في المجلدين
        img_path = os.path.join(IMG_DIR_1, img_name)
        if not os.path.exists(img_path):
            img_path = os.path.join(IMG_DIR_2, img_name)
            if not os.path.exists(img_path):
                img_path = os.path.join(BASE_DIR, img_name)

        image = Image.open(img_path).convert('RGB')
        label = int(row['label_idx'])

        if self.transform:
            image = self.transform(image)

        return image, label


# ==========================================
# 3. قراءة وتقسيم البيانات (Train / Validation)
# ==========================================
df = pd.read_csv(CSV_PATH)
df['label_idx'] = pd.Categorical(df['dx']).codes

# تقسيم البيانات 80% تدريب و 20% اختبار مع الحفاظ على نسب الأمراض (Stratify)
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_idx']
)


# ==========================================
# 4. المعالجة المسبقة وتحسين الصور (Data Augmentation)
# ==========================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = HAM10000Dataset(train_df, transform=train_transforms)
val_dataset = HAM10000Dataset(val_df, transform=val_transforms)

# تحميل البيانات بدفعات سرعة باستعمال Pin Memory
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


# ==========================================
# 5. بناء نموذج ResNet18 وتجهيز الخسارة والـ Optimizer
# ==========================================
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 7)  # تصنيف لـ 7 أنواع أمراض جلديّة
model = model.to(device)

for param in model.parameters():
    param.requires_grad = True

# موازنة الفئات غير المتكافئة (Class Weights)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_idx']),
    y=train_df['label_idx']
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=2e-5)
scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')


# ==========================================
# 6. دورة التدريب والتقييم (Training Loop)
# ==========================================
epochs = 10
best_val_acc = 0.0

print("\n Starting Fast GPU Training Now!\n")

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch [{epoch+1:02d}/{epochs}]")

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device).long()

        optimizer.zero_grad()

        # استخدام Mixed Precision لرفع السرعة وتقليل استخدام الذاكرة
        with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix(acc=f"{100. * correct / total:.1f}%")

    train_acc = 100.0 * correct / total

    # Validation Phase
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).long()

            with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
                outputs = model(images)

            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100.0 * val_correct / val_total

    print(f" Epoch [{epoch+1:02d}/{epochs}] -> Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%\n")

    # حفظ أفضل نموذج
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'resnet18_ham10000_best.pth')
        print(f" Best Model Saved! ({val_acc:.2f}%)\n")

print(f'\n Complete! Highest Accuracy: {best_val_acc:.2f}%')

 Using Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]



 Starting Fast GPU Training Now!



Epoch [01/10]: 100%|██████████| 126/126 [01:27<00:00,  1.44it/s, acc=24.2%]


 Epoch [01/10] -> Train Acc: 24.18% | Val Acc: 46.58%

 Best Model Saved! (46.58%)



...

 Epoch [08/10] -> Train Acc: 76.65% | Val Acc: 74.04%

 Best Model Saved! (74.04%)



Epoch [09/10]: 100%|██████████| 126/126 [01:18<00:00,  1.61it/s, acc=76.5%]


 Epoch [09/10] -> Train Acc: 76.55% | Val Acc: 74.19%

 Best Model Saved! (74.19%)



Epoch [10/10]: 100%|██████████| 126/126 [01:18<00:00,  1.61it/s, acc=78.1%]


 Epoch [10/10] -> Train Acc: 78.13% | Val Acc: 73.79%


 Complete! Highest Accuracy: 74.19%


In [5]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

# 1. إعداد المسارات والجهاز
BASE_DIR = '/content/ham10000'
CSV_PATH = os.path.join(BASE_DIR, 'HAM10000_metadata.csv')
IMG_DIR_1 = os.path.join(BASE_DIR, 'HAM10000_images_part_1')
IMG_DIR_2 = os.path.join(BASE_DIR, 'HAM10000_images_part_2')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using Device: {device}")

# 2. فئة الـ Dataset
class HAM10000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['image_id']
        img_name = f"{img_id}.jpg" if not str(img_id).endswith('.jpg') else img_id

        img_path = os.path.join(IMG_DIR_1, img_name)
        if not os.path.exists(img_path):
            img_path = os.path.join(IMG_DIR_2, img_name)
            if not os.path.exists(img_path):
                img_path = os.path.join(BASE_DIR, img_name)

        image = Image.open(img_path).convert('RGB')
        label = int(row['label_idx'])

        if self.transform:
            image = self.transform(image)

        return image, label

# 3. معالجة البيانات
df = pd.read_csv(CSV_PATH)
df['label_idx'] = pd.Categorical(df['dx']).codes

train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label_idx']
)

# 4. تحسين الصور بدرجة عالية وأبعاد أضخم (300x300)
IMAGE_SIZE = 300

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = HAM10000Dataset(train_df, transform=train_transforms)
val_dataset = HAM10000Dataset(val_df, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# 5. استخدام نموذج EfficientNet-B2 الأقوى والأدق
model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 7)
model = model.to(device)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_idx']),
    y=train_df['label_idx']
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

# 6. دورة التدريب (15 Epochs)
epochs = 15
best_val_acc = 0.0

print("\n Starting High-Accuracy Training (EfficientNet-B2)...\n")

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch [{epoch+1:02d}/{epochs}]")

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device).long()

        optimizer.zero_grad()
        with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix(acc=f"{100. * correct / total:.1f}%")

    train_acc = 100.0 * correct / total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).long()
            with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
                outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100.0 * val_correct / val_total
    scheduler.step(val_acc)

    print(f" Epoch [{epoch+1:02d}/{epochs}] -> Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%\n")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'efficientnet_ham10000_best.pth')
        print(f" New Best Model Saved! ({val_acc:.2f}%)\n")

print(f'\n Training Complete! Highest Accuracy Reached: {best_val_acc:.2f}%')

 Using Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 199MB/s]



 Starting High-Accuracy Training (EfficientNet-B2)...



Epoch [01/15]: 100%|██████████| 266/266 [02:32<00:00,  1.74it/s, acc=60.0%]


 Epoch [01/15] -> Train Acc: 59.95% | Val Acc: 72.59%

 New Best Model Saved! (72.59%)



...

 Epoch [13/15] -> Train Acc: 88.89% | Val Acc: 87.36%

 New Best Model Saved! (87.36%)



Epoch [14/15]: 100%|██████████| 266/266 [02:11<00:00,  2.02it/s, acc=89.2%]


 Epoch [14/15] -> Train Acc: 89.17% | Val Acc: 86.49%



Epoch [15/15]: 100%|██████████| 266/266 [02:10<00:00,  2.04it/s, acc=90.3%]


 Epoch [15/15] -> Train Acc: 90.26% | Val Acc: 88.09%

 New Best Model Saved! (88.09%)


 Training Complete! Highest Accuracy Reached: 88.09%


In [6]:
import shutil
shutil.copy('efficientnet_ham10000_best.pth', '/content/drive/MyDrive/efficientnet_ham10000_best.pth')

'/content/drive/MyDrive/efficientnet_ham10000_best.pth'

In [7]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, accuracy_score

# أسماء الأمراض الجلدية بالترتيب
class_names = [
    'akiec (سرطان حرشفي)',
    'bcc (سرطان قاعدة)',
    'bkl (ورم حميد)',
    'df (ألياف جلدية)',
    'mel (ورم ميلانيني خبيث)',
    'nv (شامة عادية)',
    'vasc (أوعية دموية)'
]

def evaluate_and_report(model, data_loader, set_name="Test"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds) * 100
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)

    print(f"\n=======================================================")
    print(f"   CLASSIFICATION REPORT FOR: {set_name.upper()} SET")
    print(f"=======================================================")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    print(f"Overall Accuracy ({set_name}): {acc:.2f}%\n")

    return report, acc

# 1. تقييم مجموعة التدريب (Train)
train_report, train_acc = evaluate_and_report(model, train_loader, set_name="Train")

# 2. تقييم مجموعة التحقق/الاختبار (Validation/Test)
val_report, val_acc = evaluate_and_report(model, val_loader, set_name="Validation/Test")

# 3. إنشاء جدول مقارنة شامل
summary_data = {
    'Dataset': ['Train Set', 'Validation / Test Set'],
    'Accuracy (%)': [f"{train_acc:.2f}%", f"{val_acc:.2f}%"],
    'Macro Avg Precision': [f"{train_report['macro avg']['precision']:.4f}", f"{val_report['macro avg']['precision']:.4f}"],
    'Macro Avg Recall': [f"{train_report['macro avg']['recall']:.4f}", f"{val_report['macro avg']['recall']:.4f}"],
    'Macro Avg F1-Score': [f"{train_report['macro avg']['f1-score']:.4f}", f"{val_report['macro avg']['f1-score']:.4f}"],
    'Weighted Avg F1-Score': [f"{train_report['weighted avg']['f1-score']:.4f}", f"{val_report['weighted avg']['f1-score']:.4f}"]
}

summary_df = pd.DataFrame(summary_data)
print("\n=== SUMMARY TABLE FOR SUPERVISOR ===")
display(summary_df)


   CLASSIFICATION REPORT FOR: TRAIN SET
                         precision    recall  f1-score   support

    akiec (سرطان حرشفي)     0.9236    1.0000    0.9603       278
      bcc (سرطان قاعدة)     0.8705    1.0000    0.9308       437
         bkl (ورم حميد)     0.9068    0.9690    0.9369       934
       df (ألياف جلدية)     0.9333    1.0000    0.9655        98
mel (ورم ميلانيني خبيث)     0.7743    0.9683    0.8605       946
        nv (شامة عادية)     0.9943    0.9249    0.9584      5699
     vasc (أوعية دموية)     0.9836    1.0000    0.9917       120

               accuracy                         0.9428      8512
              macro avg     0.9124    0.9803    0.9434      8512
           weighted avg     0.9508    0.9428    0.9443      8512

Overall Accuracy (Train): 94.28%


   CLASSIFICATION REPORT FOR: VALIDATION/TEST SET
                         precision    recall  f1-score   support

    akiec (سرطان حرشفي)     0.7818    0.8776    0.8269        49
      bcc (سرطان قاعدة)  

,Dataset,Accuracy (%),Macro Avg Precision,Macro Avg Recall,Macro Avg F1-Score,Weighted Avg F1-Score
0,Train Set,94.28%,0.9124,0.9803,0.9434,0.9443
1,Validation / Test Set,88.02%,0.8262,0.8918,0.8562,0.8841
